In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import statsmodels.api as sm

# 1. Configuration
tickers = ["RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "ICICIBANK.NS", "INFY.NS",
           "HINDUNILVR.NS", "SBIN.NS", "BHARTIARTL.NS", "ITC.NS", "KOTAKBANK.NS"]
market_ticker = "^NSEI"
all_tickers = tickers + [market_ticker]
start_date = "2014-01-01"
end_date = "2024-01-01"
rf_annual = 0.06 
# 2. Data Collection (Robust Download)
print("Downloading data...")
data = yf.download(all_tickers, start=start_date, end=end_date)

if 'Adj Close' in data.columns:
    prices = data['Adj Close']
else:
    prices = data['Close']

prices = prices.dropna(axis=1, how='all')
returns = prices.pct_change().dropna()

returns.columns = [t.replace(".NS", "") if t != "^NSEI" else "Nifty50" for t in returns.columns]
stocks = [t.replace(".NS", "") for t in tickers if t.replace(".NS", "") in returns.columns]

# 3. Part (a): Descriptive Statistics
stats = pd.DataFrame(index=returns.columns)
stats['Mean (Ann %)'] = (returns.mean() * 252 * 100).round(4)
stats['StdDev (Ann %)'] = (returns.std() * np.sqrt(252) * 100).round(4)
stats['Skewness'] = returns.skew().round(4)
stats['Kurtosis'] = returns.kurtosis().round(4)

# 4. Part (b): CAPM & IVOL
rf_daily = rf_annual / 252
ivol_results = []

for stock in stocks:
    y = returns[stock] - rf_daily
    X = returns['Nifty50'] - rf_daily
    X = sm.add_constant(X)
    
    model = sm.OLS(y, X).fit()
    
    ivol_ann = model.resid.std() * np.sqrt(252)
    ivol_results.append({
        'Stock': stock,
        'IVOL': ivol_ann,
        'Return': returns[stock].mean() * 252
    })

ivol_df = pd.DataFrame(ivol_results)
# Run this in your script to get the values for Part (b)
print(ivol_df[['Stock', 'IVOL']])
# 5. Part (c): Portfolio Sorting (Low, Medium, High IVOL)
ivol_df['Portfolio'] = pd.qcut(ivol_df['IVOL'], 3, labels=['Low', 'Medium', 'High'])
port_summary = ivol_df.groupby('Portfolio', observed=True)['Return'].mean() * 100

# 6. Part (d): Cross-sectional Regression
X_cs = sm.add_constant(ivol_df['IVOL'])
cs_model = sm.OLS(ivol_df['Return'], X_cs).fit()

print("\n--- DESCRIPTIVE STATISTICS ---")
print(stats)
print("\n--- PORTFOLIO RETURNS ---")
print(port_summary)
print("\n--- CROSS-SECTIONAL REGRESSION SUMMARY ---")
print(cs_model.summary())